# 01. Cohort Base (기본 코호트)

## 목적
슬라이딩 윈도우 적용 **전** 기본 코호트 정의

## 포함 기준
- 성인 (18세 이상)
- 첫 번째 ICU 입실
- 24시간 이상 체류

## 출력
- `cohort_base.csv`: 환자 기본 정보 (1 row = 1 patient)

## 파이프라인 위치
```
01_cohort_base.ipynb  ← 현재
    ↓
02_vital_raw.ipynb, 03_lab_raw.ipynb, ...
    ↓
10_sliding_window_merge.ipynb (슬라이딩 윈도우 + 통합)
    ↓
11_preprocessing.ipynb
    ↓
12_feature_engineering.ipynb
```

In [1]:
import duckdb
import pandas as pd
import os
from datetime import datetime

# 설정
DB_PATH = '../data/duckdb/mimic_total.duckdb'
OUTPUT_DIR = '../data/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# DuckDB 연결
con = duckdb.connect(DB_PATH)
print("=== 01. Cohort Base 생성 시작 ===")

=== 01. Cohort Base 생성 시작 ===


## Step 1: 기본 코호트 정의

In [2]:
print("Step 1: 기본 포함 기준 적용")
print("  - 성인 (18세 이상)")
print("  - 첫 번째 ICU 입실")
print("  - 24시간 이상 체류\n")

cohort_query = """
WITH ranked_stays AS (
    SELECT 
        i.subject_id,
        i.hadm_id,
        i.stay_id,
        CAST(i.intime AS TIMESTAMP) as intime,
        CAST(i.outtime AS TIMESTAMP) as outtime,
        CAST(i.los AS DOUBLE) as los,
        i.first_careunit,
        i.last_careunit,
        CAST(p.anchor_age AS INTEGER) as anchor_age,
        p.gender,
        CAST(p.dod AS TIMESTAMP) as dod,
        CAST(a.admittime AS TIMESTAMP) as admittime,
        CAST(a.dischtime AS TIMESTAMP) as dischtime,
        CAST(a.deathtime AS TIMESTAMP) as deathtime,
        a.hospital_expire_flag,
        ROW_NUMBER() OVER (PARTITION BY i.subject_id ORDER BY i.intime) as icu_seq
    FROM icustays i
    INNER JOIN patients p ON i.subject_id = p.subject_id
    INNER JOIN admissions a ON i.hadm_id = a.hadm_id
    WHERE 
        CAST(p.anchor_age AS INTEGER) >= 18
        AND CAST(i.los AS DOUBLE) >= 1.0
)
SELECT 
    subject_id,
    hadm_id,
    stay_id,
    intime,
    outtime,
    los,
    first_careunit,
    last_careunit,
    anchor_age,
    gender,
    dod,
    admittime,
    dischtime,
    deathtime,
    hospital_expire_flag,
    
    -- ICU 사망 여부
    CASE 
        WHEN deathtime IS NOT NULL AND deathtime <= outtime
        THEN 1 ELSE 0
    END as icu_mortality,
    
    -- 병원 사망 여부
    CASE 
        WHEN hospital_expire_flag = '1' THEN 1
        ELSE 0
    END as hospital_mortality
    
FROM ranked_stays
WHERE icu_seq = 1
ORDER BY stay_id
"""

df_cohort = con.execute(cohort_query).df()

print(f"✓ 기본 코호트 생성 완료")
print(f"  - 총 환자 수: {len(df_cohort):,}명")
print(f"  - ICU 사망: {df_cohort['icu_mortality'].sum():,}명 ({df_cohort['icu_mortality'].mean()*100:.2f}%)")
print(f"  - 병원 사망: {df_cohort['hospital_mortality'].sum():,}명 ({df_cohort['hospital_mortality'].mean()*100:.2f}%)")

Step 1: 기본 포함 기준 적용
  - 성인 (18세 이상)
  - 첫 번째 ICU 입실
  - 24시간 이상 체류

✓ 기본 코호트 생성 완료
  - 총 환자 수: 54,551명
  - ICU 사망: 3,837명 (7.03%)
  - 병원 사망: 5,881명 (10.78%)


## Step 2: DNR 시간 추출

In [3]:
print("\nStep 2: DNR 시간 추출")

# stay_id 리스트를 DuckDB에 등록
con.register('cohort_df', df_cohort)

dnr_query = """
SELECT 
    c.stay_id,
    MIN(CAST(ce.charttime AS TIMESTAMP)) as dnr_time
FROM cohort_df c
INNER JOIN chartevents ce ON c.stay_id = ce.stay_id
WHERE ce.itemid = '223758'  -- Code Status / DNR
GROUP BY c.stay_id
"""

df_dnr = con.execute(dnr_query).df()
print(f"✓ DNR 기록 환자: {len(df_dnr):,}명")

# 코호트에 병합
df_cohort = df_cohort.merge(df_dnr, on='stay_id', how='left')


Step 2: DNR 시간 추출
✓ DNR 기록 환자: 25,629명


## Step 3: Ventilation 시작 시간 추출

In [4]:
print("\nStep 3: Ventilation 시작 시간 추출")

vent_query = """
SELECT 
    c.stay_id,
    MIN(CAST(pe.starttime AS TIMESTAMP)) as vent_start_time
FROM cohort_df c
INNER JOIN procedureevents pe ON c.stay_id = pe.stay_id
WHERE pe.itemid = '225792'  -- Invasive Mechanical Ventilation
  AND pe.starttime IS NOT NULL
GROUP BY c.stay_id
"""

df_vent = con.execute(vent_query).df()
print(f"✓ Ventilation 기록 환자: {len(df_vent):,}명")

# 코호트에 병합
df_cohort = df_cohort.merge(df_vent, on='stay_id', how='left')


Step 3: Ventilation 시작 시간 추출
✓ Ventilation 기록 환자: 22,920명


## Step 4: Pressor 시작 시간 추출

In [5]:
print("\nStep 4: Pressor 시작 시간 추출")

pressor_query = """
SELECT 
    c.stay_id,
    MIN(CAST(ie.starttime AS TIMESTAMP)) as pressor_start_time
FROM cohort_df c
INNER JOIN inputevents ie ON c.stay_id = ie.stay_id
WHERE ie.itemid IN ('221906', '221289', '222315', '221662')  -- Norepinephrine, Epinephrine, Vasopressin, Dopamine
  AND ie.starttime IS NOT NULL
  AND ie.rate IS NOT NULL
  AND CAST(ie.rate AS DOUBLE) > 0
GROUP BY c.stay_id
"""

df_pressor = con.execute(pressor_query).df()
print(f"✓ Pressor 기록 환자: {len(df_pressor):,}명")

# 코호트에 병합
df_cohort = df_cohort.merge(df_pressor, on='stay_id', how='left')


Step 4: Pressor 시작 시간 추출
✓ Pressor 기록 환자: 11,678명


## Step 5: 결과 확인 및 저장

In [6]:
print("\n" + "="*60)
print("Cohort Base 요약")
print("="*60)

print(f"\n총 환자 수: {len(df_cohort):,}명")
print(f"\n=== 이벤트 발생 환자 ===")
print(f"  - DNR: {df_cohort['dnr_time'].notna().sum():,}명 ({df_cohort['dnr_time'].notna().mean()*100:.1f}%)")
print(f"  - Ventilation: {df_cohort['vent_start_time'].notna().sum():,}명 ({df_cohort['vent_start_time'].notna().mean()*100:.1f}%)")
print(f"  - Pressor: {df_cohort['pressor_start_time'].notna().sum():,}명 ({df_cohort['pressor_start_time'].notna().mean()*100:.1f}%)")
print(f"  - ICU 사망: {df_cohort['icu_mortality'].sum():,}명 ({df_cohort['icu_mortality'].mean()*100:.1f}%)")

print(f"\n=== 컬럼 목록 ({len(df_cohort.columns)}개) ===")
for col in df_cohort.columns:
    print(f"  - {col}")


Cohort Base 요약

총 환자 수: 54,551명

=== 이벤트 발생 환자 ===
  - DNR: 25,629명 (47.0%)
  - Ventilation: 22,920명 (42.0%)
  - Pressor: 11,678명 (21.4%)
  - ICU 사망: 3,837명 (7.0%)

=== 컬럼 목록 (20개) ===
  - subject_id
  - hadm_id
  - stay_id
  - intime
  - outtime
  - los
  - first_careunit
  - last_careunit
  - anchor_age
  - gender
  - dod
  - admittime
  - dischtime
  - deathtime
  - hospital_expire_flag
  - icu_mortality
  - hospital_mortality
  - dnr_time
  - vent_start_time
  - pressor_start_time


In [7]:
# CSV 저장
output_path = os.path.join(OUTPUT_DIR, 'cohort_base.csv')
df_cohort.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / (1024 * 1024)
print(f"\n✓ 저장 완료: cohort_base.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 행 수: {len(df_cohort):,}개 (1 row = 1 patient)")
print(f"  - 경로: {output_path}")


✓ 저장 완료: cohort_base.csv
  - 파일 크기: 12.38 MB
  - 행 수: 54,551개 (1 row = 1 patient)
  - 경로: ../data/processed/cohort_base.csv


In [8]:
# 샘플 데이터 확인
print("\n=== 샘플 데이터 (상위 5개) ===")
df_cohort.head()


=== 샘플 데이터 (상위 5개) ===


,subject_id,hadm_id,stay_id,intime,outtime,los,first_careunit,last_careunit,anchor_age,gender,dod,admittime,dischtime,deathtime,hospital_expire_flag,icu_mortality,hospital_mortality,dnr_time,vent_start_time,pressor_start_time
0,12466550,23998182,30000153,2174-09-29 12:09:00,2174-10-01 03:26:10,1.636921,Trauma SICU (TSICU),Trauma SICU (TSICU),61,M,NaT,2174-09-29 10:43:00,2174-10-15 15:24:00,NaT,0,0,0,NaT,2174-09-29 12:00:00,NaT
1,12207593,22795209,30000646,2194-04-29 01:39:22,2194-05-03 18:23:48,4.697523,Coronary Care Unit (CCU),Coronary Care Unit (CCU),43,M,2194-05-06,2194-04-27 18:43:00,2194-05-06 02:29:00,2194-05-06 02:29:00,1,0,1,2194-04-29 01:49:00,NaT,NaT
2,15726459,22744101,30000831,2140-04-17 21:26:33,2140-04-20 14:21:57,2.705139,Coronary Care Unit (CCU),Coronary Care Unit (CCU),78,M,NaT,2140-04-17 21:25:00,2140-05-18 21:00:00,NaT,0,0,0,NaT,NaT,NaT
3,12980335,23552849,30001148,2156-08-30 11:10:59,2156-08-31 14:25:34,1.135127,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),68,M,NaT,2156-08-29 11:45:00,2156-09-03 17:20:00,NaT,0,0,0,NaT,2156-08-30 14:45:00,NaT
4,12168737,29283664,30001336,2186-03-20 00:44:48,2186-03-22 19:25:44,2.778426,Coronary Care Unit (CCU),Coronary Care Unit (CCU),77,M,NaT,2186-03-20 00:44:00,2186-03-22 19:10:00,NaT,0,0,0,NaT,NaT,NaT


In [9]:
con.close()
print("\n=== 01. Cohort Base 생성 완료 ===")


=== 01. Cohort Base 생성 완료 ===
